# Complete the self-contrastive and full SEAD comparisons over ten folds and three seeds.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT
TORCH_PYTHON = RELEASE_TORCH_PYTHON
CLASSICAL_PYTHON = RELEASE_CLASSICAL_PYTHON


In [ ]:
FULL_PLAN = load_json('assets/15C/02_full_plan.json')

APPROVED_PILOT = load_json('assets/15C/02_approved_pilot.json')

SCRIPTS = {
    'source_check.py': load_text('assets/15C/workers/source_check.py'),
    'prepare_data.py': load_text('assets/15C/workers/prepare_data.py'),
    'train_sead.py': load_text('assets/15C/workers/train_sead.py'),
    'detector_core.py': load_text('assets/15C/workers/detector_core.py'),
    'select_detectors.py': load_text('assets/15C/workers/select_detectors.py'),
    'outer_prepare.py': load_text('assets/15C/workers/outer_prepare.py'),
    'outer_predict.py': load_text('assets/15C/workers/outer_predict.py'),
    'outer_metrics.py': load_text('assets/15C/workers/outer_metrics.py')
}


In [ ]:
import hashlib,json,os,subprocess,time,traceback
from pathlib import Path

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def canonical(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()).hexdigest()

def put(path,obj):
    temp=path.with_suffix('.tmp');temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(path)

def environment(executable,expected):
    probe="""import json,platform,importlib.metadata as im
v={'python':platform.python_version()}
for k in ['torch','numpy','pandas','scikit-learn','scipy','joblib']:
 try:v[k]=im.version(k)
 except im.PackageNotFoundError:v[k]=None
print(json.dumps(v))
"""
    p=subprocess.run([str(executable),'-c',probe],capture_output=True,text=True,timeout=30)
    if p.returncode:raise RuntimeError('Python metadata hatası: '+p.stderr[-2000:])
    versions=json.loads(p.stdout.strip().splitlines()[-1])
    differences={k:{'expected':v,'actual':versions.get(k)} for k,v in expected.items() if versions.get(k)!=v}
    if differences:raise RuntimeError('Çalışma ortamı değişmiş: '+json.dumps(differences,ensure_ascii=False))
    return dict(executable=str(executable),versions=versions)

def run_stage(executable,script,out,project,timeout):
    log_path=out/(script.stem+'.log')
    env=os.environ.copy();env.update(OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',PYTHONHASHSEED='42')
    with log_path.open('w',encoding='utf-8') as writer,log_path.open('r',encoding='utf-8',errors='replace') as reader:
        child=subprocess.Popen([str(executable),'-u','-X','faulthandler',str(script),str(out),str(project)],
                               stdout=writer,stderr=subprocess.STDOUT,env=env)
        started=time.monotonic()
        try:
            while child.poll() is None:
                chunk=reader.read()
                if chunk:print(chunk,end='',flush=True)
                if time.monotonic()-started>timeout:raise TimeoutError('Aşama süre sınırını aştı: '+script.name)
                time.sleep(2)
            remaining=reader.read()
            if remaining:print(remaining,end='',flush=True)
            if child.returncode:raise RuntimeError(script.name+' çıkış kodu '+str(child.returncode)+'\n'+log_path.read_text(errors='replace')[-6000:])
        except BaseException:
            if child.poll() is None:
                child.terminate()
                try:child.wait(timeout=5)
                except subprocess.TimeoutExpired:child.kill();child.wait()
            raise

import copy,hashlib,json,math,os,statistics,sys,time,traceback
from pathlib import Path


def validate_metrics(job,fold,seed):
    if job['status']!='PASS_OUTER_FOLD' or job['fold']!=fold or job['seed']!=seed:
        raise ValueError('Dış değerlendirme kimliği veya durumu uyuşmuyor.')
    expected={*(f'SCLM64/{m}' for m in ['DBSCAN','IF','LOF','OC-SVM','kNN']),
              'SEAD_adapted_raw_error','SEAD_adapted_unit_direction_control'}
    keys=[(r['method'],r['RUL_window']) for r in job['metric_rows']]
    if len(keys)!=21 or set(keys)!={(m,w) for m in expected for w in (20,30,40)}:
        raise ValueError('Dış değerlendirme yedi grup/üç referans kapsamı eksik.')
    if len(set(job['outer_engines']))!=10:raise ValueError('Dış motor sayısı10 değil.')
    for row in job['metric_rows']:
        if row['seed']!=seed or row['fold']!=fold:raise ValueError('Metrik seed/fold farklı.')
        total=sum(row[k] for k in ['TP','FP','TN','FN'])
        if total!=job['outer_rows'] or total!=row['n_outer']:raise ValueError('Confusion toplamı yanlış.')
        tp,fp,tn,fn=[row[k] for k in ['TP','FP','TN','FN']]
        ratio=lambda a,b:a/b if b else 0.
        f=ratio(2*tp,2*tp+fp+fn)
        for key,v in dict(precision=ratio(tp,tp+fp),recall=ratio(tp,tp+fn),F1=f,Macro_F1=(f+ratio(2*tn,2*tn+fp+fn))/2).items():
            if abs(row[key]-v)>1e-12:raise ValueError('Metrik sayımla uyuşmuyor: '+key)
        if not all(math.isfinite(row[k]) and 0<=row[k]<=1 for k in ['AP','ROC_AUC','F1','Macro_F1','precision','recall']):
            raise ValueError('Geçersiz metrik aralığı.')
    for name in expected:
        chosen=[r for r in job['metric_rows'] if r['method']==name]
        if len({r['threshold'] for r in chosen})!=1:raise ValueError('Duyarlılık referansı değişince eşik değiştirilmiş.')


def aggregate(jobs):
    if len(jobs)!=30 or {(j['seed'],j['fold']) for j in jobs}!={(s,f) for s in [42,43,44] for f in range(1,11)}:
        raise ValueError('Tam30koşu gereklidir; eksik koşularla nihai ortalama hesaplanmaz.')
    for seed in [42,43,44]:
        group=[j for j in jobs if j['seed']==seed]
        engines=[e for j in group for e in j['outer_engines']]
        if len(engines)!=100 or len(set(engines))!=100 or sum(j['outer_rows'] for j in group)!=20631:
            raise ValueError('Seed bazında dış motor/çevrim kapsamı yanlış.')
    for fold in range(1,11):
        matched=[j for j in jobs if j['fold']==fold]
        if len({tuple(sorted(j['outer_engines'])) for j in matched})!=1:
            raise ValueError('Seedler arasında fold motorları değişiyor.')
    rows=[r for j in jobs for r in j['metric_rows']]
    measures=['AP','ROC_AUC','precision','recall','F1','Macro_F1']
    def summarize(values):
        return {'runs':len(values),**{k+'_mean':statistics.mean(v[k] for v in values) for k in measures},
                **{k+'_descriptive_sd':statistics.stdev(v[k] for v in values) for k in measures}}
    summaries=[];per_seed=[]
    for method in sorted({r['method'] for r in rows}):
        for w in (30,20,40):
            values=[r for r in rows if r['method']==method and r['RUL_window']==w]
            summaries.append(dict(method=method,RUL_window=w,**summarize(values)))
            for seed in [42,43,44]:
                per_seed.append(dict(method=method,RUL_window=w,seed=seed,**summarize([r for r in values if r['seed']==seed])))
    paired=[]
    for j in jobs:
        values={r['method']:r for r in j['metric_rows'] if r['RUL_window']==30}
        a=values['SEAD_adapted_raw_error'];b=values['SEAD_adapted_unit_direction_control']
        paired.append(dict(seed=j['seed'],fold=j['fold'],AP_raw_minus_control=a['AP']-b['AP'],F1_raw_minus_control=a['F1']-b['F1']))
    return dict(aggregate_results=summaries,per_seed_results=per_seed,fold_results=rows,
                paired_error_definition_sensitivity=paired,
                aggregation='Equal weight per fold/seed; sample SD is descriptive. Shared folds/seeds do not form independent observations for a naive significance test.')


def verify_files(folder,record):
    for name,expected in record.items():
        path=folder/name
        if not path.is_file() or digest(path)!=expected:raise ValueError('Kayıt eksik/değişmiş: '+str(path))


def put_scripts(folder,scripts):
    folder.mkdir(parents=True,exist_ok=True)
    for name,code in scripts.items():
        path=folder/name
        if path.exists() and path.read_text()!=code:raise ValueError('Çalıştırıcı kodu değişmiş: '+str(path))
        path.write_text(code)


def completed_stage(name,executable,folder,project,files,environments,binding,timeout):
    marker=folder/(name+'_15c_completed.json')
    env=environments['torch' if name in ('train_sead','outer_predict') else 'classical']
    if marker.is_file():
        saved=json.loads(marker.read_text())
        if saved['binding']!=binding or saved['environment']!=env or set(saved['files'])!=set(files):
            raise ValueError('Aşama plan/ortam/dosya kapsamı farklı: '+str(marker))
        verify_files(folder,saved['files'])
        print('TAMAMLANMIŞ AŞAMA:',name,flush=True)
        return
    print('ÇALIŞIYOR:',name,flush=True)
    run_stage(executable,folder/(name+'.py'),folder,project,timeout)
    if not all((folder/f).is_file() for f in files):raise ValueError('Aşama çıktısı eksik: '+name)
    put(marker,dict(binding=binding,environment=env,files={f:digest(folder/f) for f in files}))


def training_stages():
    return [('prepare_data',['prepared_data.npz','prepare_summary.json']),
            ('train_sead',['training_summary.json','trained_outputs.npz','training_history.json','sclm_best.pt',
                           'sead_raw_error_best.pt','sead_unit_direction_control_best.pt']),
            ('select_detectors',['selection_summary.json','selection_scores.npz','detector_trials.json','sclm_latent_scaler.joblib']+
             ['detector_'+m.replace('-','_')+'.joblib' for m in ['DBSCAN','IF','LOF','OC-SVM','kNN']])]


def validate_selection(source):
    p=json.loads((source/'prepare_summary.json').read_text())
    t=json.loads((source/'training_summary.json').read_text())
    s=json.loads((source/'selection_summary.json').read_text())
    if p['status']!='PASS_PREPARE' or t['status']!='PASS_TRAINING' or s['status']!='PASS_SELECTION':
        raise ValueError('Eğitim/seçim onaylanmadı: '+str(source))
    if s['candidate_count']!=27 or not s['all_candidates_valid']:raise ValueError('27adayın tamamı doğrulanmadı.')
    if {r['method'] for r in s['selected_detectors']}!={'DBSCAN','IF','LOF','OC-SVM','kNN'} or len(s['selected_detectors'])!=5:
        raise ValueError('Beş-detektör kapsamı yanlış.')
    if len(s['standalone'])!=2:raise ValueError('İki SEAD kolu gerekli.')
    if len({r['engine_key'] for r in p['known_positive_samples']})!=30 or p['training_label_budget']!=30:
        raise ValueError('30motor etiket bütçesi yanlış.')
    return p,t,s


def run_full(project,torch_python,classical_python,full_plan,scripts,approved_pilot):
    project=Path(project).expanduser().absolute()
    if not project.is_dir():raise FileNotFoundError('PROJECT_DIR bulunamadı.')
    fingerprint=canonical(full_plan)
    out=project/'revision_outputs/15C_FD001_SEAD_full_cv'/fingerprint[:20]
    out.mkdir(parents=True,exist_ok=True)
    summary=out/'sead_completion_summary.json'
    status=dict(stage='15C_FD001_SEAD_full_cv',status='RUNNING',plan_sha256=fingerprint,plan=full_plan,
                expected_jobs=30,completed_jobs=0,original_detectors=['DBSCAN','IF','LOF','OC-SVM','kNN'],
                full_method_variants=['SEAD_adapted_raw_error','SEAD_adapted_unit_direction_control'],
                baseline_results_modified=False)
    lock=out/'RUNNING.lock';owns_lock=False
    try:

        fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
        with os.fdopen(fd,'w') as f:f.write(str(os.getpid()))
        owns_lock=True
        put(summary,status)
        if (out/'full_plan.json').exists() and json.loads((out/'full_plan.json').read_text())!=full_plan:
            raise ValueError('Tam CV planı değişmiş.')
        put(out/'full_plan.json',full_plan)
        base=full_plan['training_plan']
        envs={'torch':environment(torch_python,full_plan['torch_versions_frozen']),
              'classical':environment(classical_python,full_plan['classical_versions_frozen'])}
        status['environments']=envs;put(summary,status)
        for rel,expected in base['required_hashes'].items():
            if not (project/rel).is_file() or digest(project/rel)!=expected:raise ValueError('Kaynak eksik/değişmiş: '+rel)
        pilot=project/full_plan['pilot_relative_directory']
        verify_files(pilot,approved_pilot['artifact_sha256'])
        if canonical(json.loads((pilot/'plan.json').read_text()))!=approved_pilot['plan_sha256']:
            raise ValueError('Pilot planı onaylı planla uyuşmuyor.')
        if envs!=approved_pilot['environments']:raise ValueError('Pilot ile tam deney ortamları farklı.')
        jobs=[];selections=[];sources=[]
        for seed in full_plan['seeds']:
            for fold in full_plan['folds']:
                index=len(jobs)+1
                print(f'\nKOŞU {index}/30 — seed {seed}, fold {fold}',flush=True)
                folder=out/f'seed_{seed}'/f'fold_{fold:02d}'
                evaluation=folder/'evaluation'
                is_pilot=seed==42 and fold==1
                source=pilot if is_pilot else folder/'training'
                if not is_pilot:
                    put_scripts(source,{k:scripts[k] for k in full_plan['training_worker_names']})
                    plan=copy.deepcopy(base);plan.update(fold=fold,seed=seed)
                    if (source/'plan.json').exists() and json.loads((source/'plan.json').read_text())!=plan:raise ValueError('Fold eğitim planı değişmiş.')
                    put(source/'plan.json',plan)
                    binding=canonical(plan)
                    for name,files in training_stages():
                        exe=torch_python if name=='train_sead' else classical_python
                        completed_stage(name,exe,source,project,files,envs,binding,base['stage_timeout_seconds'])
                else:print('ONAYLI PİLOT EĞİTİMİ KULLANILIYOR',flush=True)
                preparation,training,selection=validate_selection(source)
                if preparation['fold']!=fold or preparation['seed']!=seed:raise ValueError('Eğitim kaynak kimliği farklı.')
                put_scripts(evaluation,scripts)
                training_files={f:digest(source/f) for _,files in training_stages() for f in files}
                context=dict(source=str(source),project=str(project),fold=fold,seed=seed,
                             training_files=training_files,full_plan_sha256=fingerprint)
                if (evaluation/'evaluation_context.json').exists() and json.loads((evaluation/'evaluation_context.json').read_text())!=context:
                    raise ValueError('Dış değerlendirme eğitim kaynakları veya yolları değişmiş.')
                put(evaluation/'evaluation_context.json',context)
                binding=canonical(context)
                stages=[('outer_prepare',classical_python,['outer_inputs.npz','outer_reference.csv','outer_prepare_summary.json']),
                        ('outer_predict',torch_python,['outer_neural_outputs.npz','outer_predict_summary.json']),
                        ('outer_metrics',classical_python,['outer_predictions.csv.gz','outer_summary.json'])]
                for name,exe,files in stages:
                    completed_stage(name,exe,evaluation,project,files,envs,binding,base['stage_timeout_seconds'])
                result=json.loads((evaluation/'outer_summary.json').read_text())
                validate_metrics(result,fold,seed)
                if digest(evaluation/'outer_predictions.csv.gz')!=result['predictions_sha256']:raise ValueError('Dış skor dosyası değişmiş.')
                jobs.append(result)
                selections.append(dict(seed=seed,fold=fold,selection=selection,training=training,
                                       known_positive_samples=preparation['known_positive_samples']))
                sources.append(dict(seed=seed,fold=fold,pilot_reused=is_pilot,source=str(source),evaluation=str(evaluation),
                                    training_hashes=training_files,outer_summary_sha256=digest(evaluation/'outer_summary.json')))
                status.update(completed_jobs=len(jobs),completed_keys=[{'seed':j['seed'],'fold':j['fold']} for j in jobs],job_sources=sources)
                put(summary,status)
        status.update(aggregate(jobs))
        status.update(status='PASS_FULL_CV',completed_jobs=30,primary_groups=7,primary_fold_rows=210,
                      total_metric_rows=630,pilot_reused=True,new_training_jobs=29,selection_records=selections,
                      predictions_retained=True,outer_evaluation_performed=True,
                      comparison_limitations=['RUL-derived late-life proxy, not annotated anomaly onset',
                        'SCLM64 uses released affine architecture and is not a dimension-matched AE ablation',
                        'Full SEAD uses30 additional fitting proxy positives; differences are not purely representational',
                        'C-MAPSS adaptation choices are explicit; not exact reproduction of published benchmark tables',
                        'No significance claim from dependent fold/seed samples'])
    except FileExistsError:
        status['status']='REVIEW_REQUIRED';status['error']='Çalışma kilidi mevcut. Aynı notebook başka bir kernelde çalışıyor olabilir: '+str(lock)

        if not owns_lock:summary=out/('sead_completion_blocked_'+str(time.time_ns())+'.json')
    except (Exception,KeyboardInterrupt):
        status['status']='REVIEW_REQUIRED';status['error']=traceback.format_exc()
    finally:
        if owns_lock:lock.unlink(missing_ok=True)
    put(summary,status)
    print('\nDURUM:',status['status'],'TAMAMLANAN:',status['completed_jobs'],'/30')
    if 'error' in status:print(status['error'])
    print('GÖNDERİLECEK DOSYA:',summary)
    return status


In [ ]:
completion_result = run_full(PROJECT_DIR, TORCH_PYTHON, CLASSICAL_PYTHON, FULL_PLAN, SCRIPTS, APPROVED_PILOT)
